In [1]:
import os
import json
import random
import shutil
from glob import glob
from PIL import Image
from sklearn.model_selection import train_test_split

base_dir = "/home/ubuntu/additional_drive/shwan_data/yolo_training/vgg_dataset/ahsan/"
output_dir = "/home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/"

img_out_train = os.path.join(output_dir, "images/train")
img_out_val = os.path.join(output_dir, "images/val")
lbl_out_train = os.path.join(output_dir, "labels/train")
lbl_out_val = os.path.join(output_dir, "labels/val")

for d in [img_out_train, img_out_val, lbl_out_train, lbl_out_val]:
    os.makedirs(d, exist_ok=True)


def get_class_dict(region_attr):
    # ✅ Accept class, classes, Classes (case insensitive)
    for key in region_attr.keys():
        if key.lower() in ["class", "classes"]:
            return region_attr[key]
    return {}


class_to_idx = {}
data_items = []

json_files = glob(os.path.join(base_dir, "*.json"))
print(f"Found JSON files: {json_files}")

for json_file in json_files:
    with open(json_file, "r") as f:
        data = json.load(f)

    project_name = os.path.splitext(os.path.basename(json_file))[0]
    img_folder = os.path.join(base_dir, project_name)

    if not os.path.isdir(img_folder):
        print(f"⚠️ Missing image folder for: {project_name}")
        continue

    meta = data["_via_img_metadata"]

    for _, item in meta.items():
        filename = item["filename"]
        image_path = os.path.join(img_folder, filename)

        if not os.path.exists(image_path):
            continue
        if len(item.get("regions", [])) == 0:
            continue

        data_items.append((image_path, item))

        # ✅ Extract classes correctly
        for region in item["regions"]:
            cls_dict = get_class_dict(region["region_attributes"])
            for cls_name in cls_dict.keys():
                if cls_name not in class_to_idx:
                    class_to_idx[cls_name] = len(class_to_idx)


print(f"\n✅ Total annotated images: {len(data_items)}")
print(f"✅ Classes Found: {class_to_idx}")

# ✅ Split train & val
train_items, val_items = train_test_split(data_items, test_size=0.20, random_state=42)


def convert_and_save(image_path, meta_item, lbl_path, out_folder):
    regions_written = 0
    width, height = Image.open(image_path).size
    lines = []

    for region in meta_item["regions"]:
        shape = region.get("shape_attributes", {})
        name = shape.get("name")

        if name not in ["polygon", "polyline"]:
            continue

        xs = shape.get("all_points_x", [])
        ys = shape.get("all_points_y", [])
        if not xs or not ys:
            continue

        cls_dict = get_class_dict(region["region_attributes"])
        if not cls_dict:
            continue

        cls_name = list(cls_dict.keys())[0]
        class_id = class_to_idx[cls_name]

        coords = " ".join([f"{x/width:.6f} {y/height:.6f}" for x, y in zip(xs, ys)])
        lines.append(f"{class_id} {coords}")
        regions_written += 1

    if regions_written > 0:
        with open(lbl_path, "w") as f:
            f.write("\n".join(lines))
        shutil.copy(image_path, out_folder)


# ✅ Train data processing
for img_path, meta_item in train_items:
    lbl_path = os.path.join(lbl_out_train, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
    convert_and_save(img_path, meta_item, lbl_path, img_out_train)

# ✅ Validation data processing
for img_path, meta_item in val_items:
    lbl_path = os.path.join(lbl_out_val, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
    convert_and_save(img_path, meta_item, lbl_path, img_out_val)


# ✅ Create YAML for YOLO
yaml_path = os.path.join(output_dir, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(f"train: images/train\n")
    f.write(f"val: images/val\n")
    f.write(f"nc: {len(class_to_idx)}\n")
    f.write("names: [" + ", ".join([f"'{c}'" for c in class_to_idx]) + "]\n")

print("\n✅ DONE — All JSONs converted successfully ✅")
print(f"✅ Classes: {class_to_idx}")
print(f"✅ data.yaml saved to: {yaml_path}")
print("✅ Ready for YOLOv11 segmentation training!")


Found JSON files: ['/home/ubuntu/additional_drive/shwan_data/yolo_training/vgg_dataset/ahsan/19_crimes_sauignon_blance.json', '/home/ubuntu/additional_drive/shwan_data/yolo_training/vgg_dataset/ahsan/19_crimes_red.json', '/home/ubuntu/additional_drive/shwan_data/yolo_training/vgg_dataset/ahsan/19_crime_rose.json', '/home/ubuntu/additional_drive/shwan_data/yolo_training/vgg_dataset/ahsan/American_Spirit_Black.json']

✅ Total annotated images: 209
✅ Classes Found: {'19_crimes_sauignon_blance': 0, '19_crimes_red_frame': 1, '19_crime_rose_frame': 2, 'American_Spirit_Black': 3}

✅ DONE — All JSONs converted successfully ✅
✅ Classes: {'19_crimes_sauignon_blance': 0, '19_crimes_red_frame': 1, '19_crime_rose_frame': 2, 'American_Spirit_Black': 3}
✅ data.yaml saved to: /home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/data.yaml
✅ Ready for YOLOv11 segmentation training!


In [3]:
# from roboflow import Roboflow
# rf = Roboflow(api_key="CQU971zsjlNYzUfRRmaC")
# project = rf.workspace("machine-learning-1-private-limited").project("products_counter-8xgul")
# version = project.version(5)
# dataset = version.download("yolov11")             

In [5]:
import os
import shutil
from glob import glob

# ✅ Main converted dataset (already created by earlier script)
main_dir = "/home/ubuntu/additional_drive/shwan_data/yolo_training/all_dataset_marge/"

main_train_img = os.path.join(main_dir, "images/train")
main_val_img = os.path.join(main_dir, "images/val")
main_train_lbl = os.path.join(main_dir, "labels/train")
main_val_lbl = os.path.join(main_dir, "labels/val")
main_yaml_path = os.path.join(main_dir, "data.yaml")

# ✅ Secondary YOLO dataset to merge
merge_dir = "/home/ubuntu/additional_drive/shwan_data/yolo_training/products_counter-5/"
merge_yaml_path = os.path.join(merge_dir, "data.yaml")

# ✅ Load MAIN YAML
with open(main_yaml_path, "r") as f:
    main_yaml = f.readlines()

for l in main_yaml:
    if l.startswith("names:"):
        main_classes = eval(l.replace("names:", "").strip())
    if l.startswith("nc:"):
        nc_main = int(l.replace("nc:", "").strip())

print("✅ Primary classes:", main_classes)

# ✅ Load SECONDARY YAML (the dataset being merged)
with open(merge_yaml_path, "r") as f:
    merge_yaml = f.readlines()

for l in merge_yaml:
    if l.startswith("names:"):
        merge_classes = eval(l.replace("names:", "").strip())
    if l.startswith("nc:"):
        nc_merge = int(l.replace("nc:", "").strip())

print("📌 Merging secondary classes:", merge_classes)

# ✅ Build global class list & remapping index
global_classes = main_classes.copy()

remap = {}
for idx, cname in enumerate(merge_classes):
    if cname not in global_classes:
        global_classes.append(cname)
    remap[idx] = global_classes.index(cname)

print("✅ Final Global class list:", global_classes)
print("✅ Remap table:", remap)

# ✅ Merge subsets
merge_sets = ["train", "valid", "test"]

def merge_subset(subset):
    src_img = os.path.join(merge_dir, subset, "images")
    src_lbl = os.path.join(merge_dir, subset, "labels")

    if not os.path.isdir(src_img):
        return

    images = glob(os.path.join(src_img, "*"))

    for img in images:
        img_name = os.path.basename(img)
        lbl_name = os.path.splitext(img_name)[0] + ".txt"
        lbl_path = os.path.join(src_lbl, lbl_name)
        if not os.path.exists(lbl_path):
            continue

        # ✅ Read + Rewrite labels with new class indexes
        new_label_lines = []
        with open(lbl_path, "r") as lbl:
            for line in lbl:
                parts = line.split()
                cid = int(parts[0])
                parts[0] = str(remap.get(cid, cid))  # ✅ proper remap
                new_label_lines.append(" ".join(parts))

        # ✅ Copy with correct destination
        if subset == "train":
            img_dst = os.path.join(main_train_img, img_name)
            lbl_dst = os.path.join(main_train_lbl, lbl_name)
        else:
            img_dst = os.path.join(main_val_img, img_name)
            lbl_dst = os.path.join(main_val_lbl, lbl_name)

        shutil.copy(img, img_dst)
        with open(lbl_dst, "w") as f:
            f.write("\n".join(new_label_lines))

        print(f"✅ Merged: {img_name}")


for s in merge_sets:
    print(f"\n📌 Merging: {s}")
    merge_subset(s)

# ✅ Update YAML
with open(main_yaml_path, "w") as f:
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write(f"nc: {len(global_classes)}\n")
    f.write("names: [" + ", ".join([f"'{c}'" for c in global_classes]) + "]\n")

print("\n✅ MERGE COMPLETED SUCCESSFULLY ✅")
print("✅ Final Classes:", global_classes)
print("✅ Total nc:", len(global_classes))
print("✅ Updated YAML:", main_yaml_path)


✅ Primary classes: ['19_crimes_sauignon_blance', '19_crimes_red_frame', '19_crime_rose_frame', 'American_Spirit_Black']
📌 Merging secondary classes: ['7_days_vanilla', '7up', 'Albanese_peach_ring', 'Bugles Carame', 'Bugles Nacho Cheese', 'Cheez it Duoz Cheddar Jack Baby Swiss', 'Cheez it Pepper Jack', 'Cheez it Snack Mix Double Cheese', 'Cheez it Snapd Extra Crunchy', 'Chex Mix Remix Buffalo Sandwich', 'Chips Ahoy Mini', 'Corn Nuts BBQ', 'Corn Nuts Chile Picante Con Limon', 'Corn Nuts Kickin Dill Pickle', 'Corn Nuts Loaded Taco', 'Corn Nuts Mexican Style Street Corn Flavor', 'Corn Nuts Nuts Original', 'Corn Nuts Ranch', 'Dots Pretzels Parmesan Garlic', 'Gardettos Chipotle Cheddar', 'Gold Fish Cheddar', 'Gold Fish Flavor Blasted Xtra Cheddar', 'Hanover Pretzel Dips Milk Chocolate', 'Hanover Pretzel Dips White Creme', 'Kettle Krinkle Cut Dill Pickle', 'Kettle Krinkle Cut Habanero Lime', 'Kettle Krinkle Cut Salt Pepper', 'Kettle Potato CHips Backyard Barbeque', 'Kettle Potato Chips Honey 

✅ Merged: frame_12300_png.rf.d56c2f40ca9907adf46bb01e17b64323.jpg
✅ Merged: frame_1020_png.rf.35c79ab5af337663bb49db1a5cb6340d.jpg
✅ Merged: altos_reposado_tequila_frame_000030_png.rf.fc44307da0f1f934e7f7d35c0838791c.jpg
✅ Merged: swisher_sweets_leaf_frame_000003_png.rf.160cb78f8b640903825c6b82c4cc453b.jpg
✅ Merged: absolute_vodka_frame_000051_png.rf.76c41784896e0aebd4e473c2f83da4d9.jpg
✅ Merged: frame_9990_png.rf.9d8935f16d9751444ea926f02f594659.jpg
✅ Merged: IMG-20251003-WA0018_jpg.rf.bdd0993672ea7a9ae142dfccaa437d7f.jpg
✅ Merged: altos_reposado_tequila_frame_000036_png.rf.b12bb6bff940742520a66b7511c4ae6e.jpg
✅ Merged: frame_45_png.rf.8df5cda1f1fa7d4e67dd5183b8154b63.jpg
✅ Merged: frame_1005_png.rf.f745eca4d016d0d7f445d25192f8a28e.jpg
✅ Merged: 7up_frame_000048_png.rf.ad1c54727865d1a21bfdfebdd4e6c477.jpg
✅ Merged: frame_3390_png.rf.140a6660eff6b363477ae2cbe04d298d.jpg
✅ Merged: frame_60_png.rf.76837f3b63fb22653c1cbdb51f97da9f.jpg
✅ Merged: frame_8130_png.rf.2873809e466c930f83827bc59a

✅ Merged: Gruet__701253006008___1_frame_018_png.rf.fd2226f8cd736c1779cf8a1737e0bc73.jpg
✅ Merged: frame_10590_png.rf.209816670d71858379c41e514ad813be.jpg
✅ Merged: twisted_tea_extreme__87692017837__4_frame_034_png.rf.69dc7bbd5ebacd94c3c7d8641f3cac78.jpg
✅ Merged: american_spirit_blue_frame_000031_png.rf.264223e6388eeb226fb8db71fa317e06.jpg
✅ Merged: altos_tequila_frame_000068_png.rf.7a3b3e3925f5eb2c6cc6f01351c12d92.jpg
✅ Merged: frame_7830_png.rf.5101159e6e01a270f0edc1c0c897b5b3.jpg
✅ Merged: american_spirit_blue_frame_000054_png.rf.eb1314c52b73cc40da19feb4c98577f8.jpg
✅ Merged: frame_13080_png.rf.a3b25c92e926a83d56a3f5659eb75cd1.jpg
✅ Merged: frame_8370_png.rf.a70c4aa1e3ca8e9ed64fabee1457bf1f.jpg
✅ Merged: frame_10050_png.rf.807e80845a7b91907db7a42316d26667.jpg
✅ Merged: frame_6360_png.rf.6444b718f25c3423d5344a20f5957470.jpg
✅ Merged: swisher_sweets_leaf_frame_000014_png.rf.5133d458a0bf4feef1d88ffee4dc36ea.jpg
✅ Merged: frame_8310_png.rf.d8183426fcb975a67b083276ea789cf2.jpg
✅ Merged: 

✅ Merged: 7_days_vanilla__48794101022__4_frame_050_png.rf.8d5beb4f3fb0bb74622de76b7d8e173c.jpg
✅ Merged: frame_4440_png.rf.f499b2b21806c71d0ee17a4056d261f2.jpg
✅ Merged: afrin_frame_000003_png.rf.bb64da8a798bc63edc3b13d6e53ad980.jpg
✅ Merged: sea_glass__85200005765__1_frame_005_png.rf.8e94d360b1e171710521cfff59fed49c.jpg
✅ Merged: frame_4230_png.rf.21ebcf4e4cef6c4b055ab28e415d6841.jpg
✅ Merged: altos_reposado_tequila_frame_000004_png.rf.7204f9ee7e3b3deb21c52428c2766fb5.jpg
✅ Merged: Albanese_peach_ring_frame_000057_png.rf.7318a7e2441f77b61eb4bbf9830a3191.jpg
✅ Merged: 7_days_vanilla__48794101022__4_frame_035_png.rf.2489bef8c17fe8e540792c34bfe7e1d2.jpg
✅ Merged: frame_5280_png.rf.c0d9a00ebd231a3632f095894187b394.jpg
✅ Merged: absolute_vodka_frame_000045_png.rf.35d1c755c0ace7dcda83b4ed01feed51.jpg
✅ Merged: frame_390_png.rf.bf1e3e78422deb98e1aea4f5d41eed5a.jpg
✅ Merged: Albanese_peach_ring_frame_000045_png.rf.3f9fa34ef3d11aec7d31cb77205ebdfd.jpg
✅ Merged: Albanese_peach_ring_frame_000058

In [1]:
pwd

'/home/ubuntu/additional_drive/shwan_data/yolo_training'